# 11. Validación de la carga en MongoDB

## Objetivo

Validar la integridad y consistencia de los documentos diarios almacenados en MongoDB Atlas después de completar la carga masiva.

La validación compara las fechas disponibles en PostgreSQL con los documentos almacenados en MongoDB, comprueba la ausencia de fechas ausentes, inesperadas o duplicadas, verifica la cobertura temporal y revisa de forma muestral la estructura de los documentos y los metadatos de sus gráficas.

In [2]:
from src.database.connection import get_database_engine
from src.database.load_available_dates import load_available_dates
from src.mongodb.connection import get_mongodb_database

In [3]:
dataset_version = "v3"
collection_name = "daily_summaries"

postgres_engine = get_database_engine()

mongodb_database, mongodb_client = get_mongodb_database()

daily_collection = mongodb_database[collection_name]

print("Conexiones establecidas correctamente.")

Conexiones establecidas correctamente.


In [4]:
available_dates = load_available_dates(
    postgres_engine
)

mongodb_documents = daily_collection.count_documents(
    {
        "dataset.version": dataset_version,
    }
)

print(f"Fechas disponibles en PostgreSQL: {len(available_dates):,}")
print(f"Documentos almacenados en MongoDB: {mongodb_documents:,}")

assert mongodb_documents == len(available_dates)

print(
    "El número de documentos en MongoDB coincide "
    "con el número de fechas disponibles en PostgreSQL."
)

Fechas disponibles en PostgreSQL: 731
Documentos almacenados en MongoDB: 731
El número de documentos en MongoDB coincide con el número de fechas disponibles en PostgreSQL.


In [5]:
from collections import Counter
from datetime import datetime

mongodb_dates_cursor = daily_collection.find(
    {
        "dataset.version": dataset_version,
    },
    {
        "_id": 0,
        "fecha": 1,
    },
).sort("fecha", 1)

mongodb_dates = [
    document["fecha"].strftime("%Y-%m-%d")
    for document in mongodb_dates_cursor
]


In [6]:
postgresql_dates_set = set(available_dates)
mongodb_dates_set = set(mongodb_dates)

missing_dates = sorted(
    postgresql_dates_set - mongodb_dates_set
)

unexpected_dates = sorted(
    mongodb_dates_set - postgresql_dates_set
)

date_counts = Counter(mongodb_dates)

duplicated_dates = sorted(
    date
    for date, count in date_counts.items()
    if count > 1
)

print(f"Fechas ausentes en MongoDB: {len(missing_dates)}")
print(f"Fechas inesperadas en MongoDB: {len(unexpected_dates)}")
print(f"Fechas duplicadas en MongoDB: {len(duplicated_dates)}")

Fechas ausentes en MongoDB: 0
Fechas inesperadas en MongoDB: 0
Fechas duplicadas en MongoDB: 0


In [7]:
assert not missing_dates, (
    f"Existen fechas de PostgreSQL ausentes en MongoDB: "
    f"{missing_dates[:10]}"
)

assert not unexpected_dates, (
    f"Existen fechas en MongoDB que no están en PostgreSQL: "
    f"{unexpected_dates[:10]}"
)

assert not duplicated_dates, (
    f"Existen fechas duplicadas en MongoDB: "
    f"{duplicated_dates[:10]}"
)

print(
    "No existen fechas ausentes, inesperadas ni duplicadas."
)

No existen fechas ausentes, inesperadas ni duplicadas.


In [8]:
print(f"Primera fecha en PostgreSQL: {available_dates[0]}")
print(f"Última fecha en PostgreSQL: {available_dates[-1]}")
print(f"Primera fecha en MongoDB: {mongodb_dates[0]}")
print(f"Última fecha en MongoDB: {mongodb_dates[-1]}")

assert mongodb_dates[0] == available_dates[0]
assert mongodb_dates[-1] == available_dates[-1]

print("El rango temporal coincide entre PostgreSQL y MongoDB.")

Primera fecha en PostgreSQL: 2023-01-01
Última fecha en PostgreSQL: 2024-12-31
Primera fecha en MongoDB: 2023-01-01
Última fecha en MongoDB: 2024-12-31
El rango temporal coincide entre PostgreSQL y MongoDB.


In [9]:
sample_indexes = [
    0,
    len(mongodb_dates) // 2,
    len(mongodb_dates) - 1,
]

sample_dates = [
    mongodb_dates[index]
    for index in sample_indexes
]

sample_datetimes = [
    datetime.strptime(date, "%Y-%m-%d")
    for date in sample_dates
]

sample_documents = list(
    daily_collection.find(
        {
            "fecha": {
                "$in": sample_datetimes,
            },
            "dataset.version": dataset_version,
        }
    )
)

print(f"Documentos seleccionados para la muestra: {len(sample_documents)}")
print(f"Fechas seleccionadas: {sample_dates}")

assert len(sample_documents) == len(sample_dates)

Documentos seleccionados para la muestra: 3
Fechas seleccionadas: ['2023-01-01', '2024-01-01', '2024-12-31']


In [10]:
required_fields = {
    "fecha",
    "dataset",
    "cobertura",
    "irradiancia",
    "meteorologia",
    "calidad",
    "procesamiento",
    "graficas",
    "metadatos",
}

for document in sample_documents:
    missing_fields = (
        required_fields - set(document.keys())
    )

    assert not missing_fields, (
        f"Faltan campos en el documento "
        f"{document.get('fecha')}: {sorted(missing_fields)}"
    )

    assert (
        document["dataset"]["version"]
        == dataset_version
    )

    assert (
        document["graficas"]
        ["curvas_solares"]
        ["disponible"]
        is True
    )

    assert (
        document["graficas"]
        ["calidad_meteorologia"]
        ["disponible"]
        is True
    )

    assert document["graficas"]["curvas_solares"]["ruta"]
    assert document["graficas"]["calidad_meteorologia"]["ruta"]

    assert (
        document["graficas"]["curvas_solares"]["formato"]
        == "png"
    )

    assert (
        document["graficas"]["calidad_meteorologia"]["formato"]
        == "png"
    )   

print(
    "La muestra de documentos contiene "
    "la estructura y las gráficas esperadas."
)

La muestra de documentos contiene la estructura y las gráficas esperadas.


In [11]:
mongodb_client.close()
postgres_engine.dispose()

print("Conexiones cerradas correctamente.")

Conexiones cerradas correctamente.
